# Silver: metadatos estáticos + enriquecimiento de cada flujo con puente_metadata

In [0]:
import dlt
from pyspark.sql.functions import col

## Metadata Puente (Static Table)

In [0]:
@dlt.table(name="02_silver.puente_metadata", comment="Metadatos estáticos para cuatro importantes puentes de sudamerica")
def puente_metadata():
    puentes = [
        {"puente_id":1, "name":"Puente Rio-Niterói", "length_m":13290, "main_span_m":300, "height_m":72, "location":"Rio de Janeiro, Brasil", "type":"Box girder bridge", "opened_year": 1974},
        {"puente_id":2, "name":"Puente Rosario-Victoria", "length_m":60000, "main_span_m":350, "height_m":50, "location":"Entre Ríos-Santa Fe, Argentina", "type":"Cable-stayed bridge complex", "opened_year": 2003},
        {"puente_id":3, "name":"Puente Centenario", "length_m":1052, "main_span_m":420, "height_m":184, "location":"Canal de Panamá, Panamá", "type":"Cable-stayed bridge","opened_year":2004},
        {"puente_id":4, "name":"Puente Chacao (en construcción)", "length_m": 2750, "main_span_m":1157, "height_m":157, "location":"Chiloé, Chile", "type":"Suspension bridge", "opened_year": 2025},
    ]
    return spark.createDataFrame(puentes)

## Temperatura Puente (Streaming Table)

In [0]:
@dlt.table(name="02_silver.puente_temperatura", comment="Temperatura enriquecida con metadatos")
@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
@dlt.expect("valid_temperature_range", "temperatura BETWEEN -20 AND 60")
def silver_puente_temperatura():
    return (
      dlt.read_stream("01_bronze.puente_temperatura")
         .withColumn("event_time", col("event_time").cast("timestamp"))
         .withColumnRenamed("puente_id", "puente_id")
         .join(dlt.read("02_silver.puente_metadata"), on="puente_id", how="left")
         .select(
           col("puente_id"), col("name"), col("location"),
           col("event_time"), col("temperatura")
         )
    )
   

## Vibración Puente (Streaming Table)

In [0]:
@dlt.table(name="02_silver.puente_vibracion", comment="Vibración enriquecida con metadatos")
@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
@dlt.expect("valid_vibration_range", "vibracion BETWEEN 0 AND 0.1")
def silver_puente_vibracion():
    return (
      dlt.read_stream("01_bronze.puente_vibracion")
         .withColumn("event_time", col("event_time").cast("timestamp"))
         .withColumnRenamed("puente_id", "puente_id")
         .join(dlt.read("02_silver.puente_metadata"), on="puente_id", how="left")
         .select(
           col("puente_id"), col("name"), col("location"),
           col("event_time"), col("vibracion")
         )
    )

## Inclinación Puente (Streaming Table)

In [0]:
@dlt.table(name="02_silver.puente_inclinacion", comment="Ángulo de inclinación enriquecido con metadatos")
@dlt.expect_or_drop("valid_event_time", "event_time IS NOT NULL")
@dlt.expect("valid_tilt_range", "inclinacion BETWEEN -0.005 AND 0.005")
def silver_puente_inclinacion():
    return (
      dlt.read_stream("01_bronze.puente_inclinacion")
         .withColumn("event_time", col("event_time").cast("timestamp"))
         .withColumnRenamed("puente_id", "puente_id")
         .join(dlt.read("02_silver.puente_metadata"), on="puente_id", how="left")
         .select(
           col("puente_id"), col("name"), col("location"),
           col("event_time"), col("inclinacion")
         )
    )